# Lab 2: Kafka + Spark Structured Streaming

## Goal

Process the same transaction stream from Lab 1, but now using Apache Spark:

- Connect Spark to Kafka,
- Apply tumbling and sliding windows,
- Stateless and stateful transformations at scale.

**Business case (continued):** Your e-commerce monitoring system needs to produce windowed reports -- e.g., total revenue per store every 5 minutes.

---

## Setup

Start the environment and make sure Kafka has the `transactions` topic from Lab 1.
Open your producer from Lab 1 in a terminal -- we'll need it generating data.

---

## Part 1: Spark Basics (Batch Mode)

Before streaming, let's verify Spark works.

### Task 1.1 -- Create a SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Lab2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

### Task 1.2 -- Create a DataFrame from sample data

Create a Spark DataFrame with the same structure as our Kafka transactions.
Use `StructType` to define the schema.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import to_timestamp

data = [
    ("TX001", "u05", 150.0, "Warsaw", "electronics", "2026-04-01 10:00:00"),
    ("TX002", "u12", 4500.0, "Krakow", "clothing", "2026-04-01 10:01:00"),
    ("TX003", "u05", 89.0, "Warsaw", "food", "2026-04-01 10:02:30"),
    ("TX004", "u08", 7800.0, "Gdansk", "electronics", "2026-04-01 10:03:00"),
    ("TX005", "u12", 55.0, "Krakow", "books", "2026-04-01 10:05:00"),
    ("TX006", "u03", 1200.0, "Wroclaw", "electronics", "2026-04-01 10:06:30"),
]

# YOUR CODE
# 1. Define schema with StructType
# 2. Create DataFrame
# 3. Convert timestamp column from string to TimestampType
# 4. Show the DataFrame and print its schema

### Task 1.3 -- Tumbling window on batch data

Group transactions into 3-minute tumbling windows. Count transactions and sum amounts per window.

In [ ]:
import pyspark.sql.functions as F

# YOUR CODE
# Use F.window("timestamp_col", "3 minutes")
# Group by window, aggregate: count, sum(amount)

---

## Part 2: Streaming from `rate` Source

Before connecting to Kafka, let's practice with the built-in `rate` source.

### Task 2.1 -- First stream

In [ ]:
batch_counter = {"count": 0}

def process_batch(df, batch_id, max_batches=5):
    batch_counter["count"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(truncate=False)
    if batch_counter["count"] >= max_batches:
        raise Exception("Stop")

# Create a rate stream (2 rows/second)
rate_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
)

rate_df.printSchema()

query = (rate_df.writeStream
    .format("console")
    .outputMode("append")
    .foreachBatch(process_batch)
    .start())

try:
    query.awaitTermination()
except:
    query.stop()

### Task 2.2 -- Enrich and filter the stream

Add columns to the rate stream:
- `user_id`: `expr("concat('u', cast(rand()*20 as int))")`
- `amount`: `expr("round(rand() * 5000, 2)")`
- `store`: `expr("case when rand() > 0.5 then 'Warsaw' else 'Krakow' end")`

Then filter only rows where `amount > 2000` (stateless, append mode).

In [ ]:
from pyspark.sql.functions import expr

batch_counter["count"] = 0

# YOUR CODE
# 1. Enrich rate_df with withColumn
# 2. Filter amount > 2000
# 3. Run stream in append mode

### Task 2.3 -- Tumbling window aggregation on stream

Group the enriched stream into 10-second tumbling windows.
Count events and sum amounts per window per store.
Use watermark of 5 seconds.

In [ ]:
batch_counter["count"] = 0

# YOUR CODE
# 1. Add watermark: .withWatermark("timestamp", "5 seconds")
# 2. Group by: window("timestamp", "10 seconds"), "store"
# 3. Aggregate: count, sum(amount)
# 4. Run in append mode (works with watermark)

---

## Part 3: Reading from Kafka

### Task 3.1 -- Connect Spark to Kafka

Start your producer from Lab 1 in a terminal, then read the stream in Spark.

In [ ]:
# Read from Kafka
kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_df.printSchema()
# Note: value is binary -- we need to decode it

### Task 3.2 -- Parse JSON from Kafka

Decode the `value` column (binary -> string -> JSON fields).

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
])

# YOUR CODE
# 1. Cast value to string: col("value").cast("string")
# 2. Parse JSON: from_json(..., tx_schema)
# 3. Select individual fields
# 4. Convert timestamp to TimestampType
# 5. Show the stream

### Task 3.3 -- Windowed aggregation on Kafka stream

On the parsed Kafka stream, compute per 1-minute tumbling window per store:
- number of transactions
- total revenue
- average transaction amount

Use a watermark of 30 seconds.

In [ ]:
# YOUR CODE

---

## Homework

1. Add a **sliding window** analysis: 5-minute window, sliding every 1 minute, per category.
2. Write the filtered alerts (amount > 3000) back to a new Kafka topic `alerts`.
   (Hint: use `.writeStream.format("kafka")`)
3. Push code to your Git repository.

**Next lab:** Decision rules and scoring on streaming data.

In [ ]:
spark.stop()